# Orbital Debris Database: Building the Analytical SQLite Source
 
**Datasets:**
- kinetic_master.csv (cleaned, merged, and physics-enriched object list)
 
**Objective:** Create a normalized SQLite database from the kinetic master file, with tables designed for efficient queries and visualizations.
 
### Why this notebook?
The project needs a single, query-ready database that brings together all cleaned and derived orbital object data. This notebook takes the master CSV and builds a normalized SQLite database for analysis and visualization.
 
### What we do here
1. **Load the master dataset:** Read in the cleaned kinetic_master.csv file.
2. **Design the schema:** Decide on tables, primary keys, and relationships for efficient queries.
3. **Clean and patch metadata:** Standardize and fill in missing values, especially for ownership and launch details.
4. **Build and export tables:** Create normalized tables and write them to SQLite.
5. **Run validation checks:** Confirm data integrity and schema alignment.
 
This sets up the foundation for all downstream queries, charts, and risk modeling.

In [1]:
import pandas as pd
import sqlite3
import utility as utils

df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

### Stage 1.0: Standardize Ownership Fields
 
**The issue:**
- Owner codes and names in the master dataset have inconsistent formatting and naming conventions, especially for major operators like SpaceX.
- Inconsistent owner fields can cause join errors and reduce data quality.
 
**What we do:**
- Strip whitespace and standardize case for `owner_code` and `owner`.
- Map common variations of SpaceX and related names to a single canonical form.
 
**Why it matters:**
- Ensures all ownership fields are consistent and ready for reliable joins and grouping in downstream tables.

In [2]:
# Standardize owner_code and owner fields
df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

df['owner'] = df['owner'].replace(owner_name_map)

# Fill missing owner with NaN if empty string or 'nan' string
df['owner'] = df['owner'].replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})

### Stage 1.1: Aggregate Ownership Metadata

**The issue:**
- Ownership metadata is spread across multiple rows and needs to be aggregated for normalization.

**What we do:**
- Aggregate ownership metadata by `owner_code`, taking the first non-null value for each string field.
- Create a unique, joinable `ownership_operators` table for the database.
- Sector flags and `users` text are derived in Stage 1.2 using mode-based logic.

**Why it matters:**
- Aggregated ownership metadata enables efficient joins and reduces redundancy in the database.

In [3]:
owner_profile = df.groupby('owner_code', as_index=False).agg({
    'owner': lambda x: x.dropna().mode().iloc[0] if not x.dropna().empty else x.name,
    'country_operator': utils.first_non_null,
    'users': utils.first_non_null,
    'contractor': utils.first_non_null,
    'contractor_country': utils.first_non_null
})

### Stage 1.2: Derive Sector Flags and Per-Object User Category

**The issue:**
- The `first_non_null` aggregation for `users` text is order-dependent and may pick an unrepresentative minority value.
- Mixed-sector owner codes (e.g., `US` covers both Starlink and NOAA) need per-object classification, not owner-level.

**What we do:**
- Compute `owner_profile.users` using the **mode** (most common non-null value) per `owner_code`.
- Derive all four sector flags (`is_commercial`, `is_government`, `is_military`, `is_civil`) from the mode-based `users` text.
- Compute a `user_category` column **per object** in `df`, using per-object UCS flags and text as the primary source, with the owner-level mode-based category as a fallback.

**Why it matters:**
- Per-object `user_category` is far more accurate than an owner-level lookup for mixed-sector owner codes.
- Storing it directly in the `satellites` table eliminates the need for join-based CASE expression in queries.

In [4]:
# Re-compute users text using mode per owner_code (more representative than first_non_null).
# first_non_null is order-dependent and can pick a minority value ('Civil' for 'US' even though 
# 'Commercial' is the majority).
users_mode = (
    df.groupby('owner_code')['users']
    .agg(lambda x: x.dropna().mode().iloc[0] if x.dropna().shape[0] > 0 else None)
    .rename('users')
)

owner_profile = owner_profile.set_index('owner_code')
owner_profile['users'] = users_mode
owner_profile = owner_profile.reset_index()

# Re-derive boolean flags from the mode-based users text.
# This eliminates the max() aggregation artifact where is_commercial=1 bleeds across
# mixed-sector owner codes like 'US'.
owner_profile['is_commercial'] = owner_profile['users'].str.contains('Commercial', case=False, na=False).astype(int)
owner_profile['is_government'] = owner_profile['users'].str.contains('Government', case=False, na=False).astype(int)
owner_profile['is_military']   = owner_profile['users'].str.contains('Military',   case=False, na=False).astype(int)
owner_profile['is_civil']      = owner_profile['users'].str.contains('Civil',      case=False, na=False).astype(int)

owner_category_map = (
    owner_profile
    .set_index('owner_code')['users']
    .map(utils.category_from_users_text)
    .to_dict()
)

# Derive user_category PER OBJECT using a three-tier priority:
df['user_category'] = df.apply(utils.derive_user_category, axis=1, map=owner_category_map)

# Quick validation
in_orbit_cat = df[df['in_orbit'] == 1]['user_category'].value_counts()
print("Per-object user_category for in-orbit objects:")
print(in_orbit_cat.to_string())
print(f"\nTotal in-orbit: {in_orbit_cat.sum():,}")


Per-object user_category for in-orbit objects:
user_category
COMMERCIAL    19296
MILITARY       7636
GOVERNMENT     6282
CIVIL           172
UNKNOWN          70

Total in-orbit: 33,456


### Stage 1.3: Patch and Fill Key Metadata
 
**The issue:**
- Some fields (e.g., `primary_purpose`, `un_registry`, `lifetime_years`, `orbit_type`, `launch_id`) are missing or inconsistent, especially for non-payload objects.
- Incomplete or inconsistent metadata can cause errors in downstream analysis and reduce data quality.
 
**What we do:**
- For non-payload objects, fill missing `primary_purpose` and `un_registry` with 'Not Applicable'.
- Ensure `lifetime_years` is numeric and nullable (no forced imputation).
- Fill missing `orbit_type` with 'Other/Misc'.
- Synthesize `launch_id` from the COSPAR prefix (YYYY-NNN), filling missing values with 'UNKNOWN'.
 
**Why it matters:**
- Ensures all key fields are complete and consistent, supporting robust queries and analysis in the final database.

In [5]:
is_payload = df['object_type'].astype(str).str.strip().str.upper().eq('PAYLOAD')

df.loc[~is_payload, 'primary_purpose'] = df.loc[~is_payload, 'primary_purpose'].fillna('Not Applicable')
df.loc[~is_payload, 'un_registry'] = df.loc[~is_payload, 'un_registry'].fillna('Not Applicable')

df['lifetime_years'] = pd.to_numeric(df['lifetime_years'], errors='coerce')
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)
df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

### Stage 2.0: Build and Export Database Tables

**The issue:**
- The master DataFrame contains all orbital object data, but analysis and visualization require normalized, query-ready tables in SQLite.
- Without normalization, queries are slow, error-prone, and difficult to maintain.

**What we do:**
- Build individual DataFrames for each logical table (satellites, orbital data, ownership, launches, etc.).
- Export each DataFrame to SQLite with schema-aligned table names.
- Run validation checks to ensure data integrity and schema alignment.

**Why it matters:**
- Normalized tables enable efficient queries, reduce redundancy, and support robust analysis and visualization.
- Validation ensures the exported database is reliable for all downstream work.

In [6]:

# build the individual tables for SQLite export, selecting relevant columns and dropping duplicates where necessary.
df_ownership_operators = owner_profile[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
 ].drop_duplicates(subset=['owner_code'])

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])

# user_category is now a per-object column derived in Stage 1.2
df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'decay_date', 'in_orbit',
     'owner_code', 'launch_id', 'user_category']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)

conn.commit()


### Stage 3: Sanity Checks and Validation

**The issue:**
- Exported tables may have missing keys, duplicates, or schema mismatches that can break downstream queries.

**What we do:**
- For each table, load from SQLite and run `utils.quick_report` to check for nulls, duplicates, and schema alignment.
- Close the database connection after validation.

**Why it matters:**
- Ensures the exported database is reliable, complete, and ready for analysis and visualization.

In [7]:
primary_keys = {
    'satellites': 'norad_id',
    'orbital_data': 'norad_id',
    'ucs_details': 'norad_id',
    'risk_assessment': 'norad_id',
    'ownership_operators': 'owner_code',
    'launch_events': 'launch_id'
}

for table, key_col in primary_keys.items():
    df_table = pd.read_sql(f"SELECT * FROM {table};", conn)
    utils.quick_report(df_table, title=f"Table: {table}", key_col=key_col)

conn.close()

print('\nSQLite build complete: ../data/clean/orbital_debris.db')

# Table: satellites

**Dimensions:** 68,437 rows × 14 columns

**Memory Footprint:** 12.29 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **cospar_id** | `str` | 0 | 100.0% | ✅ |
| **object_name** | `str` | 0 | 100.0% | ✅ |
| **satellite_name** | `str` | 60,895 | 11.0% | ⚠️ |
| **official_name** | `str` | 60,895 | 11.0% | ⚠️ |
| **object_type** | `str` | 0 | 100.0% | ✅ |
| **category** | `str` | 0 | 100.0% | ✅ |
| **ops_status** | `str` | 0 | 100.0% | ✅ |
| **data_status** | `str` | 67,179 | 1.8% | ⚠️ |
| **decay_date** | `str` | 33,456 | 51.1% | ⚠️ |
| **in_orbit** | `int64` | 0 | 100.0% | ✅ |
| **owner_code** | `str` | 0 | 100.0% | ✅ |
| **launch_id** | `str` | 0 | 100.0% | ✅ |
| **user_category** | `str` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|                |   count |   unique | top                                                                                            |   freq |
|:---------------|--------:|---------:|:-----------------------------------------------------------------------------------------------|-------:|
| cospar_id      |   68437 |    68437 | 1957-001A                                                                                      |      1 |
| object_name    |   68437 |    27351 | FENGYUN 1C DEB                                                                                 |   3531 |
| satellite_name |    7542 |     7527 | SB-WASS 3-3 (Space Based Wide Area Surveillance System) (NOSS 3-3, USA 181, NRO L23, Intruder) |      2 |
| official_name  |    7542 |     7515 | Jilin-1                                                                                        |      5 |
| object_type    |   68437 |        4 | DEBRIS                                                                                         |  35751 |
| category       |   68437 |        5 | Debris                                                                                         |  35751 |
| ops_status     |   68437 |        8 | DECAYED                                                                                        |  34981 |
| data_status    |    1258 |        2 | NEA                                                                                            |   1007 |
| decay_date     |   34981 |    14961 | 1976-08-02                                                                                     |    150 |
| owner_code     |   68437 |      129 | US                                                                                             |  27554 |
| launch_id      |   68437 |     6810 | 1999-025                                                                                       |   3537 |
| user_category  |   68437 |        5 | COMMERCIAL                                                                                     |  30781 |

### 📈 Numeric Overview
|          |   count |         mean |         std |   min |   25% |   50% |   75% |   max |
|:---------|--------:|-------------:|------------:|------:|------:|------:|------:|------:|
| norad_id |   68437 | 34230.5      | 19771.7     |     1 | 17110 | 34219 | 51331 | 68661 |
| in_orbit |   68437 |     0.488858 |     0.49988 |     0 |     0 |     0 |     1 |     1 |

# Table: orbital_data

**Dimensions:** 68,437 rows × 16 columns

**Memory Footprint:** 9.59 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **orbit_class** | `str` | 0 | 100.0% | ✅ |
| **orbit_type** | `str` | 0 | 100.0% | ✅ |
| **period_minutes** | `float64` | 0 | 100.0% | ✅ |
| **perigee_km** | `float64` | 0 | 100.0% | ✅ |
| **apogee_km** | `float64` | 0 | 100.0% | ✅ |
| **inclination_degrees** | `float64` | 0 | 100.0% | ✅ |
| **eccentricity** | `float64` | 0 | 100.0% | ✅ |
| **semi_major_axis_km** | `float64` | 0 | 100.0% | ✅ |
| **launch_mass_kg** | `float64` | 60,895 | 11.0% | ⚠️ |
| **proxy_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **dry_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **power_watts** | `float64` | 60,895 | 11.0% | ⚠️ |
| **proxy_power_watts** | `float64` | 0 | 100.0% | ✅ |
| **rcs** | `float64` | 0 | 100.0% | ✅ |
| **rcs_class** | `str` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| orbit_class |   68437 |        5 | LEO        |  61559 |
| orbit_type  |   68437 |        5 | Other/Misc |  61546 |
| rcs_class   |   68437 |        4 | UNKNOWN    |  35506 |

### 📈 Numeric Overview
|                     |   count |          mean |         std |         min |             25% |            50% |            75% |       max |
|:--------------------|--------:|--------------:|------------:|------------:|----------------:|---------------:|---------------:|----------:|
| norad_id            |   68437 | 34230.5       | 19771.7     |    1        | 17110           | 34219          | 51331          |  68661    |
| period_minutes      |   68437 |   172.274     |   676.108   |    9        |    89.58        |    94.11       |   100.22       |  95687.7  |
| perigee_km          |   68437 |  1675.62      |  6425.68    |    5        |   208           |   443          |   629          | 276715    |
| apogee_km           |   68437 |  3318.94      | 13539.5     |   46        |   284           |   484          |   833          | 807061    |
| inclination_degrees |   68437 |    68.6228    |    24.7831  |    0        |    53           |    70          |    90.25       |    150.94 |
| eccentricity        |   68437 |     0.0874561 |     5.17333 |   -0.725044 |     0.000607429 |     0.00190385 |     0.00676329 |    575    |
| semi_major_axis_km  |   68437 |  8878.84      |  9337.46    | 1433.25     |  6631.84        |  6853.58       |  7147.1        | 692996    |
| launch_mass_kg      |    7542 |   690.292     |  5358.47    |    0.5      |   148           |   260          |   280          | 450000    |
| proxy_mass_kg       |   68437 |   396.207     |  1869       |    0.5      |    50           |    50          |   355          | 450000    |
| dry_mass_kg         |   68437 |   360.611     |  1725.29    |    0.472973 |    50           |    50          |   319.5        | 420000    |
| power_watts         |    7542 |   980.728     |  2786.24    |    0        |   120           |   120          |   815          |  84000    |
| proxy_power_watts   |   68437 |   159.189     |   974.54    |    0        |     0           |     0          |   163.846      |  84000    |
| rcs                 |   68437 |     2.26835   |    13.4841  |    0.0001   |     0.01        |     0.2937     |     1          |    928.31 |

# Table: ucs_details

**Dimensions:** 68,437 rows × 7 columns

**Memory Footprint:** 5.06 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **lifetime_years** | `float64` | 60,895 | 11.0% | ⚠️ |
| **sat_age_years** | `float64` | 0 | 100.0% | ✅ |
| **primary_purpose** | `str` | 18,199 | 73.4% | ⚠️ |
| **detailed_purpose** | `str` | 60,895 | 11.0% | ⚠️ |
| **geo_longitude** | `float64` | 0 | 100.0% | ✅ |
| **un_registry** | `str` | 18,199 | 73.4% | ⚠️ |

### 📝 Object Overview
|                  |   count |   unique | top            |   freq |
|:-----------------|--------:|---------:|:---------------|-------:|
| primary_purpose  |   50238 |        8 | Not Applicable |  42696 |
| detailed_purpose |    7542 |       54 | Not Specified  |   6294 |
| un_registry      |   50238 |       68 | Not Applicable |  42696 |

### 📈 Numeric Overview
|                |   count |         mean |         std |     min |   25% |   50% |   75% |   max |
|:---------------|--------:|-------------:|------------:|--------:|------:|------:|------:|------:|
| norad_id       |   68437 | 34230.5      | 19771.7     |    1    | 17110 | 34219 | 51331 | 68661 |
| lifetime_years |    7542 |     5.14502  |     3.18728 |    0.25 |     4 |     4 |     4 |    30 |
| sat_age_years  |   68437 |    27.3801   |    20.1174  |    0    |     5 |    28 |    44 |    69 |
| geo_longitude  |   68437 |     0.189129 |     8.75422 | -179.8  |     0 |     0 |     0 |   359 |

# Table: risk_assessment

**Dimensions:** 68,437 rows × 4 columns

**Memory Footprint:** 2.09 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **velocity_kms** | `float64` | 0 | 100.0% | ✅ |
| **kinetic_joules** | `float64` | 0 | 100.0% | ✅ |
| **is_zombie** | `int64` | 0 | 100.0% | ✅ |

### 📈 Numeric Overview
|                |   count |            mean |             std |         min |             25% |             50% |             75% |             max |
|:---------------|--------:|----------------:|----------------:|------------:|----------------:|----------------:|----------------:|----------------:|
| norad_id       |   68437 | 34230.5         | 19771.7         | 1           | 17110           | 34219           | 51331           | 68661           |
| velocity_kms   |   68437 |     7.3111      |     1.06624     | 0.758409    |     7.46799     |     7.62624     |     7.75268     |    16.6766      |
| kinetic_joules |   68437 |     9.62795e+09 |     5.33148e+10 | 1.44794e+07 |     1.45234e+09 |     1.51915e+09 |     1.03124e+10 |     1.32087e+13 |
| is_zombie      |   68437 |     0.181861    |     0.385732    | 0           |     0           |     0           |     0           |     1           |

# Table: ownership_operators

**Dimensions:** 129 rows × 10 columns

**Memory Footprint:** 0.02 MB

**Primary Key Check**: ✅ No duplicate owner_code values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **owner_code** | `str` | 0 | 100.0% | ✅ |
| **owner** | `str` | 0 | 100.0% | ✅ |
| **country_operator** | `str` | 37 | 71.3% | ⚠️ |
| **users** | `str` | 37 | 71.3% | ⚠️ |
| **is_commercial** | `int64` | 0 | 100.0% | ✅ |
| **is_government** | `int64` | 0 | 100.0% | ✅ |
| **is_military** | `int64` | 0 | 100.0% | ✅ |
| **is_civil** | `int64` | 0 | 100.0% | ✅ |
| **contractor** | `str` | 37 | 71.3% | ⚠️ |
| **contractor_country** | `str` | 37 | 71.3% | ⚠️ |

### 📝 Object Overview
|                    |   count |   unique | top                 |   freq |
|:-------------------|--------:|---------:|:--------------------|-------:|
| owner_code         |     129 |      129 | AB                  |      1 |
| owner              |     129 |       90 | owner               |     37 |
| country_operator   |      92 |       74 | MULTINATIONAL       |      8 |
| users              |      92 |        9 | Commercial          |     41 |
| contractor         |      92 |       64 | Thales Alenia Space |      7 |
| contractor_country |      92 |       40 | USA                 |     23 |

### 📈 Numeric Overview
|               |   count |      mean |      std |   min |   25% |   50% |   75% |   max |
|:--------------|--------:|----------:|---------:|------:|------:|------:|------:|------:|
| is_commercial |     129 | 0.333333  | 0.473242 |     0 |     0 |     0 |     1 |     1 |
| is_government |     129 | 0.248062  | 0.433572 |     0 |     0 |     0 |     0 |     1 |
| is_military   |     129 | 0.0465116 | 0.211411 |     0 |     0 |     0 |     0 |     1 |
| is_civil      |     129 | 0.124031  | 0.330902 |     0 |     0 |     0 |     0 |     1 |

# Table: launch_events

**Dimensions:** 6,810 rows × 4 columns

**Memory Footprint:** 0.37 MB

**Primary Key Check**: ✅ No duplicate launch_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **launch_id** | `str` | 0 | 100.0% | ✅ |
| **launch_date** | `str` | 0 | 100.0% | ✅ |
| **launch_year** | `int64` | 0 | 100.0% | ✅ |
| **launch_site** | `str` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| launch_id   |    6810 |     6810 | 1957-001   |      1 |
| launch_date |    6810 |     5898 | 2022-08-04 |      5 |
| launch_site |    6810 |       67 | PLMSC      |   1551 |

### 📈 Numeric Overview
|             |   count |    mean |     std |   min |   25% |   50% |   75% |   max |
|:------------|--------:|--------:|--------:|------:|------:|------:|------:|------:|
| launch_year |    6810 | 1994.44 | 20.4241 |  1957 |  1977 |  1992 |  2015 |  2026 |


SQLite build complete: ../data/clean/orbital_debris.db
